# **Local RAG with CrewAI Knowledge and Ollama**

This quickstart shows you how to build a private RAG (Retrieval-Augmented Generation) workflow using CrewAI's Knowledge feature. No API keys needed.

Everything runs on your machine:
- **Ollama** handles text generation
- **Sentence Transformers** creates embeddings locally
- **ChromaDB** stores and retrieves your documents

We'll also add a task guardrail to validate that the agent actually uses the retrieved knowledge.

CrewAI picks the fastest available accelerator – Intel XPU, NVIDIA CUDA, Apple MPS, or CPU.

Let's get started.

## Prerequisites

Before running this notebook:

1. Install and start [Ollama](https://ollama.com), then pull a model:
   ```bash
   ollama pull llama3.2:3b
   ```
2. GPU acceleration *(optional)*: The default PyTorch install handles CUDA and CPU. For Intel XPU, install the XPU wheel:
   ```bash
   pip install torch --index-url https://download.pytorch.org/whl/xpu
   ```

In [1]:
%pip install -U --quiet crewai sentence-transformers chromadb


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Configure Your Setup

Point these at your Ollama service and preferred models. The embedding device is auto-detected — XPU, CUDA, MPS, and CPU are all supported.

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

import torch

# Suppress tracing
os.environ["CREWAI_TRACING_ENABLED"] = "false"
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

# Change these to match your setup
OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_HOST = "http://localhost:11434"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

def pick_device(requested="auto"):
    """Auto-detect the best available device: xpu, cuda, mps, or cpu."""
    if requested != "auto":
        return requested
    if hasattr(torch, "xpu") and torch.xpu.is_available():
        return "xpu"
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = pick_device()

device_info = f"Using device: {DEVICE}"
if DEVICE == "xpu":
    device_info += f" — {torch.xpu.get_device_name(0)}"
elif DEVICE == "cuda":
    device_info += f" — {torch.cuda.get_device_name(0)}"
print(device_info)

Using device: xpu — Intel(R) Arc(TM) Pro B60 Graphics


## Create a Private Knowledge Source

CrewAI's `StringKnowledgeSource` lets you feed private data directly to your agents. The text gets chunked, embedded locally with Sentence Transformers, and stored in ChromaDB. Nothing leaves your machine.

We'll use a fictional company travel policy as our private knowledge:

In [3]:
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

TRAVEL_POLICY = """Company Travel Policy (Document ID: TRAVEL-2026):
- Maximum hotel reimbursement is $250 per night.
- Flights must be booked at least 14 days in advance for reimbursement.
- Daily meal allowance is $75 per day.
- All trips require written approval from the department head before booking.
- Receipts must be submitted within 5 business days of return.
- International travel requires VP-level approval and at least 30 days advance notice."""

policy_source = StringKnowledgeSource(content=TRAVEL_POLICY)
print("Knowledge source created!")

Knowledge source created!


## Build the RAG Agent with a Guardrail

Now we create an agent that answers questions using *only* the retrieved knowledge.

We'll also add a task guardrail — a validation function that checks whether the answer includes key facts from the policy. If it fails, CrewAI retries the task with feedback telling the agent what it missed. Learn more in the [Guardrails quickstart](../Guardrails/task_guardrails.ipynb).

In [4]:
from crewai import Agent, Crew, LLM, Task, TaskOutput
from crewai.knowledge.knowledge_config import KnowledgeConfig

llm = LLM(
    model=f"ollama/{OLLAMA_MODEL}",
    base_url=OLLAMA_HOST,
    temperature=0,
)


def check_policy_answer(result: TaskOutput):
    """Validate that the answer includes key policy details."""
    required = ["$250", "14 days", "$75"]
    missing = [fact for fact in required if fact not in result.raw]
    if missing:
        return (
            False,
            f"Missing policy details: {', '.join(missing)}. "
            "Check the Additional Information and include exact values.",
        )
    return (True, result.raw)


policy_agent = Agent(
    role="Travel Policy Advisor",
    goal="Answer travel policy questions accurately using only retrieved knowledge",
    backstory=(
        "You are the company's travel desk assistant. You answer employee "
        "questions by looking up the official travel policy."
    ),
    llm=llm,
    knowledge_sources=[policy_source],
    embedder={
        "provider": "sentence-transformer",
        "config": {
            "model_name": EMBEDDING_MODEL,
            "device": DEVICE,
        },
    },
    knowledge_config=KnowledgeConfig(results_limit=3, score_threshold=0.0),
    verbose=True,
)

policy_task = Task(
    description=(
        "An employee asks: What are the hotel, flight, and meal rules "
        "in our travel policy? Answer using only the retrieved policy. "
        "Include exact dollar amounts and timeframes."
    ),
    expected_output="A clear summary of hotel, flight, and meal rules with specific numbers.",
    agent=policy_agent,
    guardrail=check_policy_answer,
    max_retries=3,
)

crew = Crew(
    agents=[policy_agent],
    tasks=[policy_task],
    tracing=False,
    verbose=True,
)

print("Crew assembled!")

Crew assembled!


## Run the Crew

When you run this cell, CrewAI will:
1. **Embed** the travel policy using Sentence Transformers on your selected device
2. **Store** the embeddings in a local ChromaDB collection
3. **Retrieve** relevant policy chunks when the agent needs them
4. **Generate** an answer with Ollama using the retrieved context
5. **Validate** the output against our guardrail

In [5]:
result = await crew.kickoff_async()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ddf98569-c958-4aef-8683-c037906ec515                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: An employee asks: What are the hotel, flight, and meal rules in our travel policy? Answer using only     │
│  the retrieved policy. Include exact dollar amounts and timeframes.                                             │
│  ID: 673004ff-95ed-4cd8-a33e-c8ba33b92eef                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔍 Knowledge Retrieval ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Knowledge Retrieval Started                                                                                    │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 📚 Knowledge Retrieved ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Search Query:                                                                                                  │
│  What are the hotel, flight, and meal rules in our travel policy?                                               │
│                                                                                                                 │
│  **Travel Policy**                                                                                              │
│                                                                                                                 │
│  **Hotel Rules**                                                                                                │
│                                                                                                                 │
│  * Employees are allowed to book hotel rooms through the company's preferred hotel chain, [Hotel Chain], at a   │
│  discounted rate of $150 per night.                                                                             │
│  * The maximum number of nights that can be booked is 5 nights.                                                 │
│  * Employees must provide proof of business-related travel when booking a hotel room.                           │
│                                                                                                                 │
│  **Flight Rules**                                                                                               │
│                                                                                                                 │
│  * Employees are entitled to reimbursement for flights booked through the company's preferred airline,          │
│  [Airline], up to $500 per trip.                                                                                │
│  * Flights must be booked at least 30 days in advance to receive reimbursement.                                 │
│  * Employees must provide receipts and proof of business-related travel when submitting flight expense          │
│  reports.                                                                                                       │
│                                                                                                                 │
│  **Meal Rules**                                                                                                 │
│                                                                                                                 │
│  * Employees are entitled to a meal allowance of $25 per day for business-related meals.                        │
│  * Meals must be purchased from approved vendors, [Vendor 1] and [Vendor 2].                                    │
│  * Employees must keep receipts and submit them with their expense reports.                                     │
│  Knowledge Retrieved:                                                                                           │
│  Additional Information: Company Travel Policy (Document ID: TRAVEL-2026):                                      │
│  - Maximum hotel reimbursement is $250 per night.                                                               │
│  - Flights must be booked at least 14 days in advance for reimbursement.                                        │
│  - Daily meal allowance is $75 per day.                                                                         │
│  - All trips require written approval from the department head before booking.                                  │
│  - Receipts must be submitted within 5 business days of return.                                                 │
│  - International travel requires VP-level approval and 

...

╭────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: def check_policy_answer(result: TaskOutput):                                                             │
│      "...                                                                                                       │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 3                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: An employee asks: What are the hotel, flight, and meal rules in our travel policy? Answer using only     │
│  the retrieved policy. Include exact dollar amounts and timeframes.                                             │
│  Agent: Travel Policy Advisor                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 3                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ddf98569-c958-4aef-8683-c037906ec515                                                                       │
│  Final Output: I can provide a summary of the hotel, flight, and meal rules as per the company's travel         │
│  policy.                                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
│  ### Hotel Reimbursement Rules:                                                                                 │
│                                                                                                                 │
│  * Maximum hotel reimbursement is $250 per night.                                                               │
│                                                                                                                 │
│  ### Flight Booking Rules:                                                                                      │
│                                                                                                                 │
│  * Flights must be booked at least 14 days in advance for reimbursement.                                        │
│                                                                                                                 │
│  ### Meal Allowance Rules:                                                                                      │
│                                                                                                                 │
│  * Daily meal allowance is $75 per day.                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [6]:
from IPython.display import display, Markdown

display(Markdown(f"### Agent's Answer\n\n{result.raw}"))

### Agent's Answer

I can provide a summary of the hotel, flight, and meal rules as per the company's travel policy.


### Hotel Reimbursement Rules:

* Maximum hotel reimbursement is $250 per night.

### Flight Booking Rules:

* Flights must be booked at least 14 days in advance for reimbursement.

### Meal Allowance Rules:

* Daily meal allowance is $75 per day.

## Conclusion

And that's it. You just ran a fully private RAG pipeline on your local machine. Here's what we covered:

- **CrewAI Knowledge** with `StringKnowledgeSource` for local document ingestion
- **Ollama** for text generation without an API key
- **Sentence Transformers** for local embeddings on your accelerator
- **Task guardrails** to validate the agent's output against source facts

No data left your machine. No API keys required.

[Learn more about CrewAI Knowledge](https://docs.crewai.com/en/concepts/knowledge)